In [1]:
import json
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

In [7]:
with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

len(cvs)

609

In [4]:
jobs = []
for cv in cvs:
    for job in cv:
        jobs.append(job)

df = pd.DataFrame(jobs)
df.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management


In [5]:
df_active = df[df["status"] == "ACTIVE"].copy()
df_active.shape

(623, 8)

In [8]:
df_active["department"].value_counts()

department
Other                     344
Information Technology     62
Sales                      46
Consulting                 39
Project Management         39
Marketing                  22
Business Development       20
Human Resources            16
Purchasing                 15
Administrative             14
Customer Support            6
Name: count, dtype: int64

In [12]:
def predict_department_baseline(title):
    
    title = title.lower()

    # Marketing
    if any(k in title for k in ["marketing", "communication", "brand", "content", "seo"]):
        return "Marketing"

    # Sales
    if any(k in title for k in ["sales", "account", "business development", "vertrieb"]):
        return "Sales"

    # Project / Consulting
    if any(k in title for k in ["project", "consultant", "beratung", "program", "pm"]):
        return "Project Management"

    # IT / Engineering
    if any(k in title for k in ["software", "developer", "engineer", "it", "data"]):
        return "IT"

    # Finance
    if any(k in title for k in ["finance", "cfo", "controller", "accounting", "buchhalter"]):
        return "Finance"

    # HR
    if any(k in title for k in ["hr", "human resources", "recruit", "people"]):
        return "HR"

    return "Other"

In [10]:
df_active["department_pred_baseline"] = df_active["position"].apply(
    predict_department_baseline
)

df_active[["position", "department", "department_pred_baseline"]].head(10)

,position,department,department_pred_baseline
0,Prokurist,Other,Other
1,CFO,Other,Finance
2,Betriebswirtin,Other,Other
3,Prokuristin,Other,Other
4,CFO,Other,Finance
6,Solutions Architect,Information Technology,IT
14,Medizintechnik Beratung,Consulting,Project Management
17,Director expansión de negocio.,Business Development,Other
18,Gerente comercial,Sales,Other
19,Administrador Unico,Administrative,Other


In [13]:
accuracy = (
    df_active["department"] == df_active["department_pred_baseline"]
).mean()

accuracy

np.float64(0.45264847512038525)

The rule-based department baseline achieves an accuracy of approximately 45%,
significantly outperforming the seniority baseline. This result is expected,
as department-specific information is often explicitly encoded in job titles,
whereas seniority signals tend to be more implicit and ambiguous.